# 02. Транспортные затраты: калькулятор фрахта и «бутылочное горлышко»

**Автор:** Степан Минаков · ветка `transport-costs` · чекпоинт 2

**Вопрос научрука:** сколько стоит вывезти карбамид с завода морем, сколько для этого нужно судов
и что ограничивает прибыль завода.

**Почему это важно для таргета.** Цена в Pink Sheet — FOB (без доставки). Покупатель платит
`CFR = FOB + фрахт`, поэтому заводу остаётся `FOB = CFR − фрахт` (netback). Рост фрахта
давит на FOB и на надбавку — значит, стоимость доставки — кандидат в признаки модели надбавки.

**Модель одного рейса** (функции в `src/logistics/freight.py`):

| Величина | Формула |
|---|---|
| время рейса `T` | погрузка + путь с грузом + выгрузка + путь порожняком + ожидание |
| аренда | `T × ставка`, ставка Supramax ≈ 3 790 + 6,09 × BDI ($/день) |
| топливо | дни × расход × цена VLSFO, цена ≈ 7,8 × Brent ($/т, оценка) |
| сборы | 2 захода в порт + Суэц (если по пути) |
| страховка | стоимость судна × ставка военной страховки |
| **$/т** | (аренда + топливо + сборы + страховка) / партия |
| **судов** | `S = Q / партия × T / 365` |

**Данные:** `data/processed/` (Pink Sheet, BDI, Comtrade) и `data/reference/` (маршруты, суда, заводы, сборы — с источниками).
Как их получить — см. `src/data/load_*.py` и `src/reference/build_routes.py`.

In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":          # запускаем из папки notebooks/ -> переходим в корень репозитория
    os.chdir("..")
sys.path.insert(0, ".")

import pandas as pd
import matplotlib.pyplot as plt
from src.logistics import freight as f

pd.set_option("display.precision", 1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"   # синий, оранжевый, бирюзовый, жёлтый
FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)

vessel = f.load_vessel("Supramax")
routes = f.load_routes()
prices = pd.read_parquet("data/processed/prices_monthly.parquet")
bdi = pd.read_parquet("data/processed/bdi_monthly.parquet")
flows = pd.read_parquet("data/processed/trade_flows.parquet")
gap = pd.read_parquet("data/processed/freight_gap.parquet")
vessel

Vessel(name='Supramax', parcel_t=47000, speed_kn=12.5, fuel_laden_tpd=28, fuel_ballast_tpd=24, fuel_port_tpd=4)

## 1. Куда везёт Катар: корзина маршрутов

Берём крупнейшего экспортёра с Ближнего Востока — Катар (QAFCO, ~6 млн т/год, порт Месаид).
Катар сам не отчитывается в Comtrade, поэтому направления берём из **зеркальных данных** (импорт покупателей).
Пять направлений дают ~80 % экспорта — их доли и будут весами корзины.

In [2]:
DEST = {"Brazil": "Paranagua (BR)", "USA": "New Orleans (US)", "Australia": "Geelong (AU)",
        "Thailand": "Laem Chabang (TH)", "India": "Kandla (IN)"}
q = flows[flows.origin == "Qatar"].groupby("dest")["t_imp"].sum()
share_total = q[list(DEST)].sum() / q.sum()
weights = (q[list(DEST)] / q[list(DEST)].sum()).rename(index=DEST)
print(f"Пять направлений = {share_total:.0%} экспорта Катара за 2015–2025")
(weights * 100).round(1).to_frame("доля, %")

Пять направлений = 80% экспорта Катара за 2015–2025


,"доля, %"
dest,
Paranagua (BR),32.7
New Orleans (US),30.1
Geelong (AU),14.1
Laem Chabang (TH),11.8
Kandla (IN),11.3


## 2. Сколько стоит рейс в условиях 2025 года

Средний BDI и Brent за 2025 год, спокойная страховка (0,06 % стоимости судна), Красное море — как в 2025 (обходят вокруг Африки).
Число судов — сколько Supramax должны постоянно работать, чтобы вывезти **весь выпуск QAFCO (6 млн т)** в пропорциях корзины.

In [3]:
BDI25 = bdi.loc["2025", "bdi_avg"].mean(); BRENT25 = prices.loc["2025", "brent"].mean()
TC25, BUNK25 = f.tc_rate(BDI25), f.bunker_price(BRENT25)
print(f"2025: BDI {BDI25:.0f} -> аренда ${TC25:,.0f}/день;  Brent ${BRENT25:.0f} -> топливо ${BUNK25:.0f}/т")

def route(dest, scenario):
    return routes[(routes.origin == "Mesaieed (QA)") & (routes.dest == dest) & (routes.scenario == scenario)].iloc[0]

QAFCO_T = 6.0e6
rows = []
for dest, w in weights.items():
    r = route(dest, "no_red_sea")      # с 2024 года суда обходят Красное море
    c = f.voyage_cost(r.distance_nm, vessel, TC25, BUNK25, via_suez=r.via_suez, war_gulf_pct=0.06)
    rows.append(dict(направление=dest, вес=w, миль=r.distance_nm, **c,
                     судов=f.fleet_needed(QAFCO_T * w, c["T_days"], vessel)))
snap = pd.DataFrame(rows).set_index("направление")
basket25 = (snap.total * snap.вес).sum()
print(f"Корзина Катара, 2025: ${basket25:.1f}/т;  судов для 6 млн т: {snap.судов.sum():.0f}")
snap.round(1)

2025: BDI 1662 -> аренда $13,912/день;  Brent $69 -> топливо $539/т
Корзина Катара, 2025: $39.8/т;  судов для 6 млн т: 24


,вес,миль,T_days,hire,fuel,fees,war,total,судов
направление,,,,,,,,,
Paranagua (BR),0.3,8499,70.7,20.9,17.5,2.8,0.3,41.5,8.1
New Orleans (US),0.3,12237,95.6,28.3,24.9,2.8,0.3,56.3,10.1
Geelong (AU),0.1,6814,59.4,17.6,14.2,2.8,0.3,34.8,2.9
Laem Chabang (TH),0.1,4477,43.8,13.0,9.5,2.8,0.3,25.5,1.8
Kandla (IN),0.1,1129,21.5,6.4,2.9,2.8,0.3,12.3,0.9


**Что видно:** стоимость тонны растёт почти пропорционально расстоянию — от ~$12 до Индии до ~$50+ до США.
Чтобы вывозить 6 млн т в такой пропорции, нужно около 25 Supramax, постоянно занятых только на Катар.

**Важное свойство модели:** если разделить годовые затраты на объём, объём `Q` сокращается —
**стоимость тонны не зависит от выпуска завода**, она задаётся длиной рейса, ставкой аренды и топливом.
Выпуск определяет только **число судов**. Удельная стоимость растёт от объёма лишь тогда, когда флот на рынке
заканчивается и ставки поднимаются — это видно в BDI, а не в объёме конкретного завода.

## 3. Помесячный ряд `transport_cost_per_t` (2015 – 2026)

Для каждого месяца: аренда из BDI, топливо из Brent, военная страховка и маршрут — по фактическому режиму:

| Период | Красное море | Страховка в заливе (% стоимости судна) | Источник |
|---|---|---|---|
| до 12.2023 | открыто (в США — через Суэц) | 0,06 | S&P Global, июнь 2025 |
| 01.2024 – 02.2026 | обход вокруг Африки | 0,06 | там же |
| 03 – 06.2026 | обход | 2 *(допущение: «1–3 %» за несколько недель до 22.07)* | S&P Global, июль 2026 |
| с 07.2026 | обход | 8,75 (середина 7,5–10) | S&P Global, июль 2026 |

Это **признак для модели надбавки** к чекпоинту 3 — сохраняем его в `data/processed/transport_cost_monthly.parquet`.

In [4]:
def gulf_war_pct(d):
    if d < pd.Timestamp("2026-03-01"): return 0.06
    if d < pd.Timestamp("2026-07-01"): return 2.0
    return 8.75

def basket_cost(d, bdi_v, brent_v):
    scen = "normal" if d < pd.Timestamp("2024-01-01") else "no_red_sea"
    tc, bunk, war = f.tc_rate(bdi_v), f.bunker_price(brent_v), gulf_war_pct(d)
    parts = {k: 0.0 for k in ["hire", "fuel", "fees", "war", "total", "T_days"]}
    for dest, w in weights.items():
        r = route(dest, scen)
        c = f.voyage_cost(r.distance_nm, vessel, tc, bunk, via_suez=r.via_suez, war_gulf_pct=war)
        for k in parts: parts[k] += w * c[k]
    return parts

m = bdi[["bdi_avg"]].join(prices[["brent", "urea_price"]], how="inner").loc["2015-04":"2026-09"]   # период для склейки: апрель 2015 — сентябрь 2026
ts = pd.DataFrame([basket_cost(d, r.bdi_avg, r.brent) for d, r in m.iterrows()], index=m.index)
ts = ts.rename(columns={"total": "transport_cost_per_t"}).join(m)
# для склейки с признаками команды: date — колонка (первое число месяца), только наши признаки, без чужих brent / urea_price
FEATURES = ["transport_cost_per_t", "hire", "fuel", "fees", "war", "T_days", "bdi_avg"]
ts[FEATURES].rename_axis("date").reset_index().to_parquet("data/processed/transport_cost_monthly.parquet", index=False)
print(f"{len(ts)} месяцев, {ts.index.min():%Y-%m} — {ts.index.max():%Y-%m}")
ts[["bdi_avg", "brent", "hire", "fuel", "fees", "war", "transport_cost_per_t"]].resample("YE").mean().round(1)

138 месяцев, 2015-04 — 2026-09


,bdi_avg,brent,hire,fuel,fees,war,transport_cost_per_t
date,,,,,,,
2015-12-31,750.9,51.8,11.1,11.3,4.7,0.3,27.4
2016-12-31,671.4,44.1,10.5,9.6,4.7,0.3,25.0
2017-12-31,1128.8,54.4,14.2,11.9,4.7,0.3,31.0
2018-12-31,1336.6,71.1,15.9,15.5,4.7,0.3,36.3
2019-12-31,1333.9,64.0,15.8,14.0,4.7,0.3,34.8
2020-12-31,1074.6,42.3,13.7,9.2,4.7,0.3,27.9
2021-12-31,2921.9,70.5,28.7,15.4,4.7,0.3,49.0
2022-12-31,1865.2,99.8,20.1,21.8,4.7,0.3,46.9
2023-12-31,1401.6,82.6,16.4,18.0,4.7,0.3,39.4


In [5]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.stackplot(ts.index, ts.hire, ts.fuel, ts.fees, ts.war, colors=[C1, C2, C3, C4],
             labels=["аренда судна", "топливо", "сборы", "военная страховка"], alpha=0.9)
ax.set_ylabel("$ за тонну"); ax.set_xlim(ts.index.min(), ts.index.max())
ax.set_title("Стоимость доставки карбамида из Катара (корзина 5 направлений), $/т")
ax.legend(loc="upper left", frameon=False, ncol=2)
for x, t in [("2021-10-01", "бум фрахта"), ("2024-01-01", "обход\nКрасного моря"), ("2026-07-01", "Ормуз:\nстраховка 7,5–10 %")]:
    ha = "right" if x.startswith("2026") else "center"
    ax.annotate(t, (pd.Timestamp(x), ts.transport_cost_per_t.asof(pd.Timestamp(x))), xytext=(-6 if ha == "right" else 0, 28),
                textcoords="offset points", ha=ha, fontsize=8, color="#52514e",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
                arrowprops=dict(arrowstyle="-", color="#9b9a95", lw=0.8))
ax.set_ylim(0, ts.transport_cost_per_t.max() * 1.25)
fig.tight_layout(); fig.savefig(FIG / "transport_cost_monthly.png", dpi=200); plt.show()

/var/folders/qy/6zz6xqkx36g2l46ljqw84bj40000gn/T/ipykernel_9604/4007178996.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "transport_cost_monthly.png", dpi=200); plt.show()


## 4. Проверка на реальных данных: модель против Comtrade (CIF − FOB)

Comtrade даёт фактическую разницу между ценой у покупателя (CIF) и у продавца (FOB) — это фрахт + страховка.
Медиана по ~100 маршрутам мира в год. Уровни сравнивать напрямую нельзя (у Катара маршруты длиннее среднего),
но **динамика должна совпадать**.

In [6]:
yr = ts.transport_cost_per_t.resample("YE").mean(); yr.index = yr.index.year
cmp_ = pd.concat([yr.rename("модель, Катар"), gap.freight_gap_median.rename("Comtrade CIF−FOB, медиана")], axis=1).dropna()
corr = cmp_.corr().iloc[0, 1]
print(f"Корреляция по годам 2015–2025: {corr:.2f};  отношение уровней: {(cmp_.iloc[:,0]/cmp_.iloc[:,1]).median():.2f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(cmp_.index, cmp_.iloc[:, 0], color=C1, lw=2, marker="o", ms=5, label="модель: корзина Катара")
ax.plot(cmp_.index, cmp_.iloc[:, 1], color=C2, lw=2, marker="o", ms=5, label="факт: Comtrade CIF − FOB (медиана мира)")
ax.set_ylabel("$ за тонну"); ax.set_ylim(0); ax.legend(frameon=False, loc="upper left")
ax.set_title(f"Модель и фактическая стоимость доставки по годам (корреляция {corr:.2f})")
fig.tight_layout(); fig.savefig(FIG / "model_vs_comtrade.png", dpi=200); plt.show()
cmp_.round(1).T

Корреляция по годам 2015–2025: 0.89;  отношение уровней: 1.36


/var/folders/qy/6zz6xqkx36g2l46ljqw84bj40000gn/T/ipykernel_9604/2946430311.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "model_vs_comtrade.png", dpi=200); plt.show()


,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
"модель, Катар",27.4,25.0,31.0,36.3,34.8,27.9,49.0,46.9,39.4,43.2,39.8
"Comtrade CIF−FOB, медиана",21.9,17.3,21.1,23.8,23.1,22.2,51.1,49.8,31.1,29.5,29.3


## 5. Сценарии: что двигает стоимость доставки

База — условия 2025 года. Меняем по одному фактору. Открытие Красного моря влияет только на маршрут в США (через Суэц короче на ~2 700 миль): Индия и Бразилия идут без него.

In [7]:
def scen_cost(dest, bdi_v=BDI25, brent_v=BRENT25, red_sea_open=False, war=0.06, wait=0):
    r = route(dest, "normal" if red_sea_open else "no_red_sea")
    return f.voyage_cost(r.distance_nm, vessel, f.tc_rate(bdi_v), f.bunker_price(brent_v),
                         via_suez=r.via_suez, war_gulf_pct=war, extra_wait_days=wait)["total"]

SCEN = {
    "база 2025": {},
    "Красное море открыто": dict(red_sea_open=True),
    "BDI ×2 (как 2021)": dict(bdi_v=2 * BDI25),
    "Brent $120": dict(brent_v=120),
    "очереди +10 дней": dict(wait=10),
    "страховка Ормуз 8,75 %": dict(war=8.75),
}
sc = pd.DataFrame({name: {d: scen_cost(d, **kw) for d in weights.index} for name, kw in SCEN.items()}).T
sc["корзина"] = sc[weights.index] @ weights
sc = sc[["Kandla (IN)", "Paranagua (BR)", "New Orleans (US)", "корзина"]]

fig, ax = plt.subplots(figsize=(9, 3.8))
y = range(len(sc)); h = 0.38
ax.barh([i + h/2 for i in y], sc["Kandla (IN)"], h, color=C1, label="Катар → Индия (короткий)")
ax.barh([i - h/2 for i in y], sc["Paranagua (BR)"], h, color=C2, label="Катар → Бразилия (длинный)")
ax.set_yticks(list(y), sc.index); ax.invert_yaxis(); ax.set_xlabel("$ за тонну")
for i, (a, b) in enumerate(zip(sc["Kandla (IN)"], sc["Paranagua (BR)"])):
    ax.text(a + 0.8, i + h/2, f"{a:.0f}", va="center", fontsize=8); ax.text(b + 0.8, i - h/2, f"{b:.0f}", va="center", fontsize=8)
ax.set_xlim(0, sc[["Kandla (IN)", "Paranagua (BR)"]].values.max() * 1.45)
ax.legend(frameon=False, loc="center right"); ax.set_title("Сценарии: стоимость доставки, $/т")
fig.tight_layout(); fig.savefig(FIG / "scenarios.png", dpi=200); plt.show()
sc.round(1)

/var/folders/qy/6zz6xqkx36g2l46ljqw84bj40000gn/T/ipykernel_9604/4003457917.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "scenarios.png", dpi=200); plt.show()


,Kandla (IN),Paranagua (BR),New Orleans (US),корзина
база 2025,12.3,41.5,56.3,39.8
Красное море открыто,12.3,41.5,52.0,38.5
BDI ×2 (как 2021),16.9,56.7,76.8,54.4
Brent $120,14.4,54.4,74.7,52.1
очереди +10 дней,15.7,44.9,59.7,43.2
"страховка Ормуз 8,75 %",49.3,78.4,93.2,76.8


## 6. Где «бутылочное горлышко»

Проверяем три кандидата: флот (судов не хватит?), порт (успеет ли погрузить?) и пролив.

In [8]:
S = snap.судов.sum()
load_days_year = QAFCO_T / f.DEFAULTS["load_rate"]
yanbu_trucks_month = 4.8e6 / 12 / 20          # SABIC: вся продукция по суше в Янбу, грузовики по 20 т
print(f"1) Флот: QAFCO нужно ~{S:.0f} Supramax. Мировой флот Supramax/Ultramax — тысячи судов: для одного завода флот не горлышко.")
print(f"2) Порт: погрузка 6 млн т при {f.DEFAULTS['load_rate']:,.0f} т/сут занимает {load_days_year:.0f} причало-дней в год -> нужно ≥{load_days_year/365:.1f} причала.")
print(f"3) Пролив: при закрытом Ормузе Катару обойти нечем. SABIC возит часть по суше в Янбу: "
      f"вывезти весь выпуск (4,8 млн т) так = ~{yanbu_trucks_month:,.0f} грузовиков в месяц.")
war_share = ts.loc["2026-07":, "war"].mean() / ts.loc["2026-07":, "transport_cost_per_t"].mean()
print(f"   С июля 2026 военная страховка = {war_share:.0%} стоимости доставки.")

1) Флот: QAFCO нужно ~24 Supramax. Мировой флот Supramax/Ultramax — тысячи судов: для одного завода флот не горлышко.
2) Порт: погрузка 6 млн т при 12,500 т/сут занимает 480 причало-дней в год -> нужно ≥1.3 причала.
3) Пролив: при закрытом Ормузе Катару обойти нечем. SABIC возит часть по суше в Янбу: вывезти весь выпуск (4,8 млн т) так = ~20,000 грузовиков в месяц.
   С июля 2026 военная страховка = 39% стоимости доставки.


## Выводы

1. **Стоимость доставки из Катара в 2025 году — около $40/т** в среднем по направлениям: от ~$12 (Индия) до ~$57 (США).
   Динамика совпадает с фактической разницей CIF − FOB из Comtrade (корреляция 0,89 по годам); уровень у Катара
   в ~1,4 раза выше мировой медианы, потому что его маршруты длиннее среднего.
2. **Стоимость тонны не зависит от выпуска завода** — её задают длина рейса, ставка аренды (BDI) и топливо (Brent).
   Выпуск определяет только число судов (~24 Supramax на 6 млн т QAFCO).
3. **В спокойные годы главные драйверы — ставка аренды (BDI) и топливо (Brent).** В 2021 году рост дал аренда,
   в 2022 — топливо: поэтому одним BDI 2022 год не объяснялся, а с топливом модель его ловит.
4. **В 2026 году «бутылочное горлышко» — Ормузский пролив, а не флот и не фрахт.** Военная страховка выросла
   с 0,06 % до 7,5–10 % стоимости судна, это +~$37/т к каждому рейсу (~40 % всей стоимости доставки с июля 2026).
   А обход по суше (SABIC → Янбу) упирается в тысячи грузовиков в месяц, то есть возможен только для части объёма.
5. Ряд `transport_cost_per_t` сохранён как **признак для модели надбавки** (чекпоинт 3).

## Ограничения
- Ставка аренды из BDI калибрована по 3 точкам; цена топлива — грубый пересчёт из Brent.
- Сборы, скорость погрузки, сбор Суэца — оценки (`is_estimate` в `data/reference/`).
- Страховка в марте–июне 2026 — допущение (2 %); месяцы 2026 года без данных о реальном трафике через Ормуз.
- Следующий шаг: добавить трафик проливов из PortWatch (Ормуз, Баб-эль-Мандеб) — задержки и долю заблокированного объёма.